<a href="https://colab.research.google.com/github/Clanboy777/THE-OP-BANK-OF-6-7/blob/main/FACEREAL_org.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U "transformers>=5.10.1" accelerate flask flask-cors requests

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("Loading Alpha AI...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

model.eval()

print("✅ Alpha AI loaded!")
print("GPU:", torch.cuda.is_available())

Loading Alpha AI...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

✅ Alpha AI loaded!
GPU: True


In [3]:
messages = [
    {
        "role": "user",
        "content": "Hello! Introduce yourself as Alpha AI."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("🤖 Alpha AI:")
print(answer)

🤖 Alpha AI:
Hello! I'm Alpha AI, a highly advanced language model designed to assist and engage in conversations. I'm here to provide information, answer questions to the best of my ability, and help with various tasks. I'm always ready to learn and improve, so feel free to guide me or correct me if I make a mistake. I'm excited to assist you!


In [4]:
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import threading
import time
import requests

app = Flask(__name__)
CORS(app)

chat_history = []

SYSTEM_PROMPT = """You are Alpha AI, a helpful and friendly AI assistant.

Give clear, useful and simple answers.

Your creator is Akshat.

If someone asks who made you, who created you, who built you,
who programmed you, or who your creator is, answer exactly:

I was made by Akshat. 🤖
"""

def generate_response(user_message):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    messages.extend(chat_history[-10:])

    messages.append({
        "role": "user",
        "content": user_message
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return answer.strip()

print("✅ Alpha AI engine ready!")

✅ Alpha AI engine ready!


In [5]:
import threading
import time
import requests
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

chat_history = []

SYSTEM_PROMPT = """You are Alpha AI, a helpful and friendly AI assistant.

Give clear, useful and simple answers.

Your creator is Akshat.

If someone asks who made you, who created you, who built you,
who programmed you, or who your creator is, answer exactly:

I was made by Akshat. 🤖
"""

def generate_response(user_message):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    messages.extend(chat_history[-10:])

    messages.append({
        "role": "user",
        "content": user_message
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return answer.strip()

print("✅ Alpha AI engine ready!")

@app.route("/chat", methods=["POST"])
def chat():

    try:
        data = request.get_json()
        message = data.get("message", "").strip()

        if not message:
            return jsonify({"response": "Please type a message."})

        lower = message.lower()
        print(f"DEBUG: User message (lowercase): {lower}")

        creator_questions = [
            "who made you",
            "who created you",
            "who built you",
            "who programmed you",
            "who is your creator",
            "who is your maker"
        ]

        lord_questions = [
            "who is the dangerous man with some money in his pocket?",
            "who is the god of music",
            "who is the goat of music?",
            "who is the best musician ever?",
            "who is the 24K singer?",
            "who is the best singer?",
            "WHO is the best multi-instrumentalist ever?"
        ]

        goat_questions = [
            "who is bruno mars?"
        ]

        # Prioritize hardcoded responses
        if any(q in lower for q in creator_questions):
            answer = "I was made by Akshat. 🤖"
            print("DEBUG: Matched creator question.")
        elif any(q in lower for q in lord_questions):
            answer = "BRUNO MARS. 🤖"
            print("DEBUG: Matched lord question.")
        elif any(q in lower for q in goat_questions):
            answer = """Peter Gene Hernandez, known universally as **Bruno Mars**, stands as one of the most complete entertainers, songwriters, and vocalists of the modern era. Blending Motown showmanship, classic 1980s funk, early-2000s R&B, and modern pop, Mars has bridged generational gaps to build a catalog with over 150 million records sold worldwide.

---

**Early Roots and Musical Upbringing**

Born on October 8, 1985, in Honolulu, Hawaii, Mars was raised in a musical family. His father, Peter Hernandez, was a Latin percussionist, and his mother, Bernadette San Pedro Bayot, was a singer and dancer. Performing from the age of four in his family's review show, *The Love Notes*, he gained early fame across Hawaii as an Little Elvis impersonator.

This early exposure to classic rock, doo-wop, soul, and reggae gave Mars an intuitive mastery of stage presence, rhythmic timing, and vocal control before he reached adolescence.

---

**The Smeezingtons and Breakthrough**

After graduating high school, Mars moved to Los Angeles in 2003 to pursue a music career. Following an early record deal with Motown that fell through, he pivoted to songwriting and production. Alongside Philip Lawrence and Ari Levine, Mars formed **The Smeezingtons**, a production trio that authored major hits for other artists, including:

* **"Right Round"** by Flo Rida
* **"Billionaire"** by Travie McCoy (featuring Bruno Mars)
* **"Nothin' on You"** by B.o.B (featuring Bruno Mars)

"Nothin' on You" and "Billionaire" showcased Mars's melodic hooks to global audiences in 2010, establishing his voice on pop radio and setting up his solo debut.

---

**Discography and Career Evolution**

| Year | Title | Signature Tracks | Cultural Impact |
| --- | --- | --- | --- |
| **2010** | *Doo-Wops & Hooligans* | "Just the Way You Are", "Grenade", "The Lazy Song" | Established Mars as a premier pop-balladeer; certified multi-platinum globally. |
| **2012** | *Unorthodox Jukebox* | "Locked Out of Heaven", "When I Was Your Man", "Treasure" | Expanded into reggae-rock, disco, and 80s pop synth-work; won Best Pop Vocal Album at the Grammys. |
| **2016** | *24K Magic* | "24K Magic", "That's What I Like", "Versace on the Floor" | Dedicated entirely to 80s/90s R&B, retro-funk, and new jack swing; swept the 60th Grammy Awards (including Album, Record, and Song of the Year). |
| **2021** | *An Evening with Silk Sonic* (with Anderson .Paak) | "Leave the Door Open", "Smokin Out the Window" | A retro 70s soul side-project that swept 4 Grammy Awards. |

---

**Key Collaborations and Event Highlights**

Beyond his solo studio releases, Mars has delivered defining moments in live performance and collaborative pop history:

* **"Uptown Funk" (2014):** Created with producer Mark Ronson, the track spent 14 weeks at #1 on the Billboard Hot 100, won Record of the Year at the Grammys, and became one of the best-selling digital singles of all time.
* **Super Bowl Halftime Performances:** Mars headlined Super Bowl XLVIII in 2014, drawing over 115 million viewers. He returned as a special guest for Super Bowl 50 in 2016 alongside Beyoncé and Coldplay.
* **"Die with a Smile" (2024):** A pop ballad duet with Lady Gaga that earned widespread chart success and critical praise.
* **"APT." (2024):** A pop-punk/pop collaboration with ROSÉ of BLACKPINK that reached international chart peaks.

---

**Artistry and Showmanship**

Mars's distinct place in modern music stems from his complete technical skill set:

* **Vocal Range:** A tenor voice capable of clean head voice transitions, full-chested high notes, and expressive R&B runs.
* **Multi-Instrumental Talent:** Proficient in playing the drums, guitar, bass, and keyboards both in studio arrangements and live on stage.
* **The Hooligans:** Mars performs alongside his live band, **The Hooligans**, who execute choreographed routines, horn sections, and dynamic tempo changes without reliance on heavy playback tracks.

Through a commitment to classical musicianship, vintage arrangement techniques, and energetic stagecraft, Bruno Mars remains a central figure in modern pop and R&B music.
. 🤖"""
            print("DEBUG: Matched og question.")
        else:
            answer = generate_response(message)
            print("DEBUG: Falling back to general generation.")

        chat_history.append({
            "role": "user",
            "content": message
        })

        chat_history.append({
            "role": "assistant",
            "content": answer
        })

        return jsonify({"response": answer})

    except Exception as e:
        print("ERROR:", e)
        return jsonify({
            "response": "Error: " + str(e)
        }), 500


@app.route("/reset", methods=["POST"])
def reset():

    chat_history.clear()

    return jsonify({
        "success": True
    })


html = """
<!DOCTYPE html>
<html>

<head>

<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>PROJECT ALPHA</title>

<style>

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    background: #0f172a;
    color: white;
    font-family: Arial, sans-serif;

    height: 100vh;

    display: flex;
    justify-content: center;
    align-items: center;
}

.container {
    width: 95%;
    max-width: 850px;
    height: 90vh;

    background: #1e293b;

    border-radius: 18px;

    display: flex;
    flex-direction: column;

    overflow: hidden;
}

.header {
    padding: 18px;

    background: #334155;

    display: flex;
    justify-content: space-between;
    align-items: center;

    font-size: 21px;
    font-weight: bold;
}

.newchat {
    background: #475569;
    color: white;

    border: none;
    border-radius: 8px;

    padding: 10px 14px;

    cursor: pointer;
}

.messages {
    flex: 1;

    padding: 20px;

    overflow-y: auto;
}

.message {
    padding: 12px 15px;

    margin-bottom: 12px;

    border-radius: 12px;

    max-width: 75%;

    white-space: pre-wrap;

    line-height: 1.5;
}

.user {
    background: #2563eb;
    margin-left: auto;
}

.bot {
    background: #475569;
}

.input-area {
    display: flex;

    gap: 10px;

    padding: 15px;

    background: #334155;
}

input {
    flex: 1;

    padding: 13px;

    border: none;

    outline: none;

    border-radius: 8px;

    font-size: 16px;
}

.send {
    background: #2563eb;

    color: white;

    border: none;

    border-radius: 8px;

    padding: 0 20px;

    cursor: pointer;
}
H6{color:violet; position:fixed;
                 right:10px;
                bottom:5px;}

.abc{color:purple;}

</style>

</head>

<body>

<div class="container">

<div class="header">

<div>🤖 SHADOW'S ALPHA AI</div>

<button class="newchat" onclick="newChat()">
🔄 New Chat
</button>

</div>

<div id="messages" class="messages">

<div class="message bot">
Hello! 👋 I'm Alpha AI. How can I help you?
</div>

</div>

<div class="input-area">

<input
id="input"
placeholder="Message Alpha AI..."
autocomplete="off"
>

<button class="send" onclick="sendMessage()">
Send
</button>

</div>

</div>
<h6>Created by Akshat Mishra <i class="abc">(SHADOW)</i></H6>

<script>

const input = document.getElementById("input");
const messages = document.getElementById("messages");

input.addEventListener("keydown", function(event) {

    if (event.key === "Enter") {
        sendMessage();
    }

});

function addMessage(text, type) {

    const div = document.createElement("div");

    div.className = "message " + type;

    div.textContent = text;

    messages.appendChild(div);

    messages.scrollTop = messages.scrollHeight;
}

async function sendMessage() {

    const text = input.value.trim();

    if (!text) return;

    addMessage(text, "user");

    input.value = "";

    const loading = document.createElement("div");

    loading.className = "message bot";

    loading.textContent = "Alpha AI is thinking...";

    messages.appendChild(loading);

    try {

        const response = await fetch("/chat", {

            method: "POST",

            headers: {
                "Content-Type": "application/json"
            },

            body: JSON.stringify({
                message: text
            })

        });

        const data = await response.json();

        loading.remove();

        addMessage(
            data.response || "No response.",
            "bot"
        );

    } catch (error) {

        loading.remove();

        addMessage(
            "❌ Connection error.",
            "bot"
        );

        console.error(error);
    }
}

async function newChat() {

    await fetch("/reset", {
        method: "POST"
    });

    messages.innerHTML = "";

    addMessage(
        "New chat started! 👋",
        "bot"
    );
}

</script>

</body>

</html>
"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)

@app.route("/")
def home():
    return send_file("index.html")

print("✅ Website and API created!")

✅ Alpha AI engine ready!
✅ Website and API created!


In [6]:
def run_server():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

test = requests.get(
    "http://127.0.0.1:5000",
    timeout=10
)

print("Status:", test.status_code)

if test.status_code == 200:
    print("✅ Alpha AI website is working!")
    print("Local: http://127.0.0.1:5000")
else:
    print("❌ Flask problem")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 14:54:47] "GET / HTTP/1.1" 200 -


Status: 200
✅ Alpha AI website is working!
Local: http://127.0.0.1:5000


In [7]:
import subprocess
import re
import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Starting public link...")

cloudflare = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:5000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

while True:

    line = cloudflare.stdout.readline()

    if not line:
        continue

    print(line.strip())

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:

        print()
        print("====================================")
        print("🌍 ALPHA AI IS LIVE!")
        print("====================================")
        print(match.group(0))
        print("====================================")

        break

Starting public link...
2026-09-02T14:54:48Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-02T14:54:48Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-02T14:54:54Z INF +--------------------------------------------------------------------------------------------+
2026-09-02T14:54:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-02T14:54:54Z INF |  https://scotland-brilliant-po

In [8]:
import requests

print("Testing Alpha AI server...")

try:
    r = requests.get("http://127.0.0.1:5000", timeout=10)
    print("Status:", r.status_code)

    if r.status_code == 200:
        print("✅ Flask is running correctly!")
    else:
        print("❌ Flask returned:", r.status_code)

except Exception as e:
    print("❌ Flask is NOT running!")
    print("Error:", e)

INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 14:54:54] "GET / HTTP/1.1" 200 -


Testing Alpha AI server...
Status: 200
✅ Flask is running correctly!


In [9]:
# Stop old tunnel
try:
    cloudflare.terminate()
except:
    pass

time.sleep(2)

# Start a fresh tunnel
cloudflare = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:5000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Starting new tunnel...\n")

while True:

    line = cloudflare.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            print("\n================================")
            print("🌍 ALPHA AI PUBLIC LINK")
            print("================================")
            print(match.group(0))
            print("================================")
            break

⏳ Starting new tunnel...

2026-09-02T14:54:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-02T14:54:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-02T14:55:01Z INF +--------------------------------------------------------------------------------------------+
2026-09-02T14:55:01Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-02T14:55:01Z INF |  https://manufacture-change-